In [ ]:
from pathlib import Path
import shutil
import yaml

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
BASE_DIR = Path.cwd().parent.resolve()  # Ejecutar desde pec3/notebooks

DATA_DIR = BASE_DIR / "data"

ORAL_FINAL_DIR = DATA_DIR / "oral_seg_final"

YOLO_DIR = BASE_DIR / "yolo" / "oral_seg_yolo"

YOLO_IMAGES_DIR = YOLO_DIR / "images"
YOLO_LABELS_DIR = YOLO_DIR / "labels"

OUTPUT_DIR = BASE_DIR / "outputs"
TABLES_DIR = OUTPUT_DIR / "tables"
FIGURES_DIR = OUTPUT_DIR / "figures"

CSV_FINAL = TABLES_DIR / "oral_seg_dataset_final.csv"

for carpeta in [YOLO_IMAGES_DIR, YOLO_LABELS_DIR, TABLES_DIR, FIGURES_DIR]:
    carpeta.mkdir(parents=True, exist_ok=True)

print("CSV_FINAL existe:", CSV_FINAL.exists())
print("YOLO_DIR:", YOLO_DIR)

In [ ]:
if YOLO_DIR.exists():
    shutil.rmtree(YOLO_DIR)

for split in ["train", "val", "test"]:
    (YOLO_IMAGES_DIR / split).mkdir(parents=True, exist_ok=True)
    (YOLO_LABELS_DIR / split).mkdir(parents=True, exist_ok=True)

print("Dataset YOLO limpio.")

In [ ]:
df = pd.read_csv(CSV_FINAL)

print("Total muestras:", len(df))

print("\nSplit:")
print(df["split"].value_counts())

print("\nOrigen:")
print(df["origen"].value_counts())

print("\nClase lengua:")
print(df["clase_lengua"].value_counts())

display(df.head())

In [ ]:
def mascara_a_poligono_yolo(mask_path, area_min=100):
    """
    Convierte una máscara binaria a una línea YOLO segmentation.
    
    Formato YOLO segmentation:
    class_id x1 y1 x2 y2 x3 y3 ...
    
    Coordenadas normalizadas entre 0 y 1.
    """
    
    mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
    
    if mask is None:
        return None
    
    h, w = mask.shape[:2]
    
    mask_bin = (mask > 0).astype(np.uint8) * 255
    
    contornos, _ = cv2.findContours(
        mask_bin,
        cv2.RETR_EXTERNAL,
        cv2.CHAIN_APPROX_SIMPLE
    )
    
    if len(contornos) == 0:
        return None
    
    # Nos quedamos con el contorno más grande para evitar ruido pequeño
    contornos = sorted(contornos, key=cv2.contourArea, reverse=True)
    contorno = contornos[0]
    
    area = cv2.contourArea(contorno)
    
    if area < area_min:
        return None
    
    # Simplificar un poco el contorno para que no sea enorme
    epsilon = 0.002 * cv2.arcLength(contorno, True)
    contorno_aprox = cv2.approxPolyDP(contorno, epsilon, True)
    
    puntos = contorno_aprox.reshape(-1, 2)
    
    if len(puntos) < 3:
        return None
    
    valores = ["0"]  # clase 0 = lengua
    
    for x, y in puntos:
        x_norm = x / w
        y_norm = y / h
        
        x_norm = min(max(x_norm, 0.0), 1.0)
        y_norm = min(max(y_norm, 0.0), 1.0)
        
        valores.append(f"{x_norm:.6f}")
        valores.append(f"{y_norm:.6f}")
    
    linea = " ".join(valores)
    
    return linea

In [ ]:
fila = df.iloc[0]

mask_path = fila["mask_final"]
linea = mascara_a_poligono_yolo(mask_path)

print("Mask:", mask_path)
print("Línea YOLO:")
print(linea[:300] if linea is not None else None)

In [ ]:
filas_yolo = []
errores = []

for i, fila in df.iterrows():
    split = fila["split"]
    
    img_path = Path(fila["img_final"])
    mask_path = Path(fila["mask_final"])
    
    if not img_path.exists():
        errores.append({
            "id_global": fila["id_global"],
            "motivo": "imagen_no_existe"
        })
        continue
    
    if not mask_path.exists():
        errores.append({
            "id_global": fila["id_global"],
            "motivo": "mascara_no_existe"
        })
        continue
    
    linea_yolo = mascara_a_poligono_yolo(mask_path)
    
    if linea_yolo is None:
        errores.append({
            "id_global": fila["id_global"],
            "motivo": "no_se_pudo_crear_poligono"
        })
        continue
    
    nombre_base = img_path.stem
    
    img_out = YOLO_IMAGES_DIR / split / f"{nombre_base}.jpg"
    label_out = YOLO_LABELS_DIR / split / f"{nombre_base}.txt"
    
    shutil.copy2(img_path, img_out)
    
    with open(label_out, "w", encoding="utf-8") as f:
        f.write(linea_yolo + "\n")
    
    filas_yolo.append({
        "id_global": fila["id_global"],
        "split": split,
        "origen": fila["origen"],
        "clase_lengua": fila["clase_lengua"],
        "img_yolo": str(img_out),
        "label_yolo": str(label_out),
        "mask_final": str(mask_path),
        "filename_original": fila["filename_original"]
    })

df_yolo = pd.DataFrame(filas_yolo)
df_errores = pd.DataFrame(errores)

print("Muestras YOLO creadas:", len(df_yolo))
print("Errores:", len(df_errores))

display(df_yolo.head())

if len(df_errores) > 0:
    display(df_errores.head())

In [ ]:
csv_yolo = TABLES_DIR / "oral_seg_yolo_dataset.csv"
csv_errores = TABLES_DIR / "oral_seg_yolo_prepare_errores.csv"

df_yolo.to_csv(csv_yolo, index=False)
df_errores.to_csv(csv_errores, index=False)

print("CSV YOLO:", csv_yolo)
print("CSV errores:", csv_errores)

In [ ]:
yaml_path = YOLO_DIR / "oral_seg_yolo.yaml"

data_yaml = {
    "path": str(YOLO_DIR),
    "train": "images/train",
    "val": "images/val",
    "test": "images/test",
    "names": {
        0: "lengua"
    }
}

with open(yaml_path, "w", encoding="utf-8") as f:
    yaml.dump(data_yaml, f, sort_keys=False, allow_unicode=True)

print("YAML guardado en:")
print(yaml_path)

print("\nContenido:")
print(data_yaml)

In [ ]:
for split in ["train", "val", "test"]:
    n_img = len(list((YOLO_IMAGES_DIR / split).glob("*.jpg")))
    n_lab = len(list((YOLO_LABELS_DIR / split).glob("*.txt")))
    
    print(split)
    print(" imágenes:", n_img)
    print(" labels:", n_lab)

In [ ]:
def leer_label_yolo(label_path, ancho, alto):
    """
    Lee un txt YOLO segmentation y devuelve puntos en píxeles.
    """
    
    with open(label_path, "r", encoding="utf-8") as f:
        lineas = f.readlines()
    
    if len(lineas) == 0:
        return []
    
    partes = lineas[0].strip().split()
    
    coords = partes[1:]
    coords = [float(c) for c in coords]
    
    puntos = []
    
    for i in range(0, len(coords), 2):
        x = int(coords[i] * ancho)
        y = int(coords[i + 1] * alto)
        puntos.append([x, y])
    
    return np.array(puntos, dtype=np.int32)

In [ ]:
muestras = df_yolo.sample(
    n=min(8, len(df_yolo)),
    random_state=7
).reset_index(drop=True)

plt.figure(figsize=(12, 8))

for i, fila in muestras.iterrows():
    img = cv2.imread(fila["img_yolo"], cv2.IMREAD_COLOR)
    
    if img is None:
        continue
    
    alto, ancho = img.shape[:2]
    
    puntos = leer_label_yolo(fila["label_yolo"], ancho, alto)
    
    overlay = img.copy()
    
    if len(puntos) >= 3:
        puntos_cv = puntos.reshape((-1, 1, 2))
        cv2.fillPoly(overlay, [puntos_cv], (0, 120, 255))
        img = cv2.addWeighted(overlay, 0.35, img, 0.65, 0)
    
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    
    plt.subplot(2, 4, i + 1)
    plt.imshow(img_rgb)
    plt.title(fila["origen"] + " " + fila["clase_lengua"], fontsize=8)
    plt.axis("off")

plt.tight_layout()

fig_path = FIGURES_DIR / "oral_seg_yolo_labels_muestras.png"
plt.savefig(fig_path, dpi=150)
plt.show()

print("Figura guardada:", fig_path)